# Qwen3.5-9B · 학습 640² / 추론 672² · loss 비교

이번 실험은 해상도를 확정한 뒤 **loss 대상만 변경**합니다.

| 조건 | 학습 | 추론 | 감독 대상 |
|---|---|---|---|
| full_text | 640² | 672² | 원본 baseline 전체 입력, padding 제외 |
| answer_only | 640² | 672² | 정답 문자 + 바로 뒤 종료 토큰 |

두 조건 모두 원본 Qwen3.5-9B에서 새 LoRA를 학습합니다. 기존 384 학습 LoRA를 이어 학습하거나 기준 점수로 재사용하지 않습니다.
기존 baseline 출력에서 고정 학습/검증 ID, seed, 프롬프트, LR 1e-4, 1 epoch, NF4, LoRA 설정을 가져옵니다.
**기존 93.2%는 학습 384²의 결과이며 이번 full_text 640² 결과와 구분합니다.**

### 실행
1. baseline을 실행한 프로젝트에서 이 노트북을 열고 BASELINE_RUN_DIR을 지정합니다(후보 하나면 자동 선택).
2. Run All: 준비 → 전체 입력 loss 학습/평가 → 정답 loss 학습/평가 → 비교표.
3. 학습 중 `supervised_tokens.csv`를 확인하면 실제 loss 대상 토큰을 볼 수 있습니다.

각 조건은 1회 새 학습과 검증 약 500개 추론을 수행하므로 **새 학습 총 2회**입니다.
완료 학습은 재사용합니다. 학습 도중 중단하면 해당 학습만 처음부터 다시 시작합니다.
정답 span은 실제 템플릿의 정답 포함/빈 assistant 입력을 비교해 찾습니다. 정답 디코딩과 종료 토큰 검증이 실패하면 자동 추정하지 않고 중단합니다.
평가는 동일 프롬프트/파싱 정책입니다. dev/test/최종 holdout 평가나 자동 제출은 없습니다.
loss 정의가 다르므로 학습 loss 숫자 대신 생성 Accuracy와 개선/악화 문항으로 판단합니다.
loss 변경은 02 독립 검토 대상이며 현재 검토 미완료입니다.


## 1. 기준 실행 폴더 지정
다른 컴퓨터에서 파일을 복사할 필요는 없습니다. 해당 컴퓨터에서 완료한 baseline 출력 폴더를 사용합니다.

In [1]:
from pathlib import Path
import os, sys, json, hashlib, subprocess, time
PROJECT_DIR = Path.cwd().resolve()
BASELINE_RUN_DIR = None  # 예: PROJECT_DIR / "output/TASK-006/TASK006-xxxxxxxxxxxxxxxx"
ENV_PYTHON = PROJECT_DIR / "downloads/envs/TASK006_baseline_qwen35" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
INFERENCE_RESOLUTION = 672  # 고정 추론 해상도
RESOLUTIONS = ["full_text", "answer_only"]  # 비교할 loss 조건
SESSION_TAG = "loss_640_672_v1"
RETRY_OOM = False  # 이미 OOM으로 기록된 조건도 재시도하려면 True

if INFERENCE_RESOLUTION is None:
    raise RuntimeError("추론 해상도 미정입니다. 결과 확인 후 INFERENCE_RESOLUTION에 선택값을 입력하세요.")
if not isinstance(INFERENCE_RESOLUTION,int) or isinstance(INFERENCE_RESOLUTION,bool) or INFERENCE_RESOLUTION<=0:
    raise ValueError("INFERENCE_RESOLUTION은 양의 정수여야 합니다.")
if BASELINE_RUN_DIR is None:
    candidates = sorted(p.parent for p in (PROJECT_DIR / "output/TASK-006").glob("TASK006-*/lora_eval_complete.json"))
    if len(candidates) != 1:
        print("baseline 후보:")
        for p in candidates: print(p)
        raise RuntimeError("BASELINE_RUN_DIR에 비교할 baseline 결과 폴더를 지정하세요.")
    BASELINE_RUN_DIR = candidates[0]
BASELINE_RUN_DIR = Path(BASELINE_RUN_DIR).resolve()
for name in ["run_config.json","task006_worker.py","audit_complete.json","train_complete.json","lora_eval_complete.json","requirements.lock.txt","model_assets.json"]:
    if not (BASELINE_RUN_DIR / name).is_file(): raise FileNotFoundError(BASELINE_RUN_DIR / name)
if not ENV_PYTHON.is_file(): raise FileNotFoundError(f"baseline 전용 Python 경로를 확인하세요: {ENV_PYTHON}")
print("baseline:",BASELINE_RUN_DIR)
print("학습 640² / 추론 672², loss 비교:",RESOLUTIONS)


baseline: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006\TASK006-8f2c71505c11284c
학습 640² / 추론 672², loss 비교: ['full_text', 'answer_only']


## 2. 실행 함수 정의
아래 코드는 별도 GPU 프로세스로 실행됩니다. 기존 checkpoint와 분할을 수정하지 않습니다.

In [2]:
RESOLUTION_WORKER = r'''
import csv, hashlib, importlib.util, json, os, sys, time, uuid
from pathlib import Path


def digest(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()


def verify_worker_source(path, expected):
    data=Path(path).read_bytes()
    raw=hashlib.sha256(data).hexdigest()
    lf=hashlib.sha256(data.replace(b"\r\n",b"\n")).hexdigest()
    if expected not in (raw,lf):
        raise RuntimeError(f"baseline 코드 내용이 다릅니다. expected={expected}, raw={raw}, LF={lf}. 기존 파일/설정을 수정하지 말고 실행 폴더를 확인하세요.")
    return expected


def read(path): return json.loads(Path(path).read_text(encoding='utf-8'))


def write(path,obj):
    p=Path(path);tmp=p.with_name(p.name+'.tmp')
    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str),encoding='utf-8');tmp.replace(p)


def verified_status(root,size):
    p=root/f'res_{size}_status.json'
    if not p.exists(): return {'state':'not_run'}
    rec=read(p)
    if rec['state']=='completed':
        for name,sha in rec['artifacts'].items():
            f=Path(rec['directory'])/name
            if not f.is_file() or digest(f)!=sha: raise RuntimeError(f'완료 결과 변경/누락: {f}')
    return rec


def paired(a,b):
    m=a.merge(b,on='id',validate='one_to_one',suffixes=('_full_text','_new'))
    if len(m)!=len(a) or len(m)!=len(b) or not (m.gold_full_text==m.gold_new).all():
        raise RuntimeError('비교 ID/정답이 동일하지 않습니다.')
    if not (m.group_id_full_text==m.group_id_new).all(): raise RuntimeError('이미지 그룹 불일치')
    old=m.answer_full_text==m.gold_full_text;new=m.answer_new==m.gold_new
    m['transition']=['gain' if y and not x else 'loss' if x and not y else 'same_correct' if x else 'same_wrong' for x,y in zip(old,new)]
    return m,{'gain':int((~old & new).sum()),'loss':int((old & ~new).sum()),
              'delta_pp':100*float(new.mean()-old.mean())}


def summarize(cfg):
    import pandas as pd
    root=Path(cfg['output_dir']);rows=[];preds={}
    for mode in cfg['resolutions']:
        rec=verified_status(root,mode)
        r={'loss_mode':mode,'status':rec['state'],'train_resolution':640,'inference_resolution':672}
        if rec['state']=='completed':
            d=Path(rec['directory']);r.update(read(d/'valid_metrics.json'))
            r['accuracy_pct']=100*r['accuracy']
            preds[mode]=pd.read_csv(d/'valid_predictions.csv',keep_default_na=False)
        else:r['error']=rec.get('error','')
        rows.append(r)
    if len(preds)==2:
        m,stats=paired(preds['full_text'],preds['answer_only'])
        m.to_csv(root/'paired_predictions.csv',index=False,encoding='utf-8-sig')
        m[m.transition.isin(['gain','loss'])].to_csv(root/'changed_answers.csv',index=False,encoding='utf-8-sig')
        rows[1].update(stats)
        types=[]
        for mode,p in preds.items():
            for name,g in p.groupby('question_type'):
                types.append({'loss_mode':mode,'question_type':name,'n':len(g),'accuracy':float((g.answer==g.gold).mean())})
        pd.DataFrame(types).to_csv(root/'type_comparison.csv',index=False,encoding='utf-8-sig')
    table=pd.DataFrame(rows);table.to_csv(root/'loss_comparison.csv',index=False,encoding='utf-8-sig')
    write(root/'summary.json',{'results':rows,'adoption':'pending','review_02':'pending','test_used':False,'holdout_evaluated':False})
    (root/'PROJECT_STATUS_update.md').write_text('# TASK-006 loss 비교\n\n학습 640² / 추론 672². 기존 384 학습 점수와 직접 비교하지 않음.\n\n'+table.to_string(index=False)+'\n\n독립 검토/채택 미완료.',encoding='utf-8')
    print(table.to_string(index=False),flush=True)
    return rows


def load_baseline(cfg):
    base=Path(cfg['baseline_dir']);original=read(base/'run_config.json')
    worker=base/'task006_worker.py'
    if verify_worker_source(worker,original['worker_sha256'])!=cfg['baseline_worker_sha256']:
        raise RuntimeError('baseline 실행 코드 해시 불일치')
    spec=importlib.util.spec_from_file_location('saved_baseline',worker)
    module=importlib.util.module_from_spec(spec);sys.modules[spec.name]=module;spec.loader.exec_module(module)
    module.block_network()
    return module,original


def prepare(cfg):
    import pandas as pd
    root=Path(cfg['output_dir']);base=Path(cfg['baseline_dir']);module,original=load_baseline(cfg)
    train=module.completed(base,'train');evaluation=module.completed(base,'lora_eval')
    audit=module.completed(base,'audit')
    if train is None or evaluation is None or audit is None:raise RuntimeError('baseline audit/train/lora_eval을 먼저 완료하세요.')
    if original['pixel_budget']!=384**2:raise RuntimeError('학습 해상도 384²인 baseline을 지정하세요.')
    reload_check=read(Path(train['directory'])/'reload_check.json')
    if not reload_check.get('identical_outputs'):raise RuntimeError('baseline 저장/재로드 검증 미통과')
    _,valid,info=module.load_split(original,base)
    # Avoid relying only on paths; check every model asset against baseline download hashes.
    assets=read(base/'model_assets.json')
    if assets['revision']!=original['revision']:raise RuntimeError('모델 revision 불일치')
    for name,record in assets['files'].items():
        f=Path(original['model_dir'])/name
        if not f.is_file() or digest(f)!=record['sha256']:raise RuntimeError(f'모델 파일 확인 필요: {name}')
    valid[['id','group_id','question_type']].to_csv(root/'fixed_valid_ids.csv',index=False)
    frozen={'baseline_config':original,'data_manifest':info,'checkpoint':str(Path(train['directory'])/'adapter_epoch1'),
            'baseline_records':{name:digest(base/(name+'_complete.json')) for name in ['audit','train','lora_eval']},
            'baseline_config_sha256':digest(base/'run_config.json'),'valid_ids_sha256':digest(root/'fixed_valid_ids.csv'),
            'valid_n':len(valid),'previous_predictions':str(Path(evaluation['directory'])/'valid_lora_predictions.csv'),
            'model_file_stats':{name:[(Path(original['model_dir'])/name).stat().st_size,(Path(original['model_dir'])/name).stat().st_mtime_ns] for name in assets['files']}}
    frozen_path=root/'frozen_inputs.json'
    if frozen_path.exists() and read(frozen_path)!=frozen:raise RuntimeError('기준 입력 변경. 새 SESSION_TAG로 실행하세요.')
    write(frozen_path,frozen)
    print('준비 완료. 저장 LoRA:',frozen['checkpoint'],'검증:',len(valid),flush=True)
    summarize(cfg)


def configure_pixels(adapter,pixels):
    ip=adapter.processor.image_processor
    ip.size={'shortest_edge':pixels,'longest_edge':pixels}
    if hasattr(ip,'min_pixels'):ip.min_pixels=pixels
    if hasattr(ip,'max_pixels'):ip.max_pixels=pixels


def validate_inputs(cfg):
    import pandas as pd
    root=Path(cfg['output_dir']);base=Path(cfg['baseline_dir']);frozen=read(root/'frozen_inputs.json')
    module,original=load_baseline(cfg)
    if digest(base/'run_config.json')!=frozen['baseline_config_sha256']:raise RuntimeError('baseline 설정 변경')
    for name,sha in frozen['baseline_records'].items():
        if digest(base/(name+'_complete.json'))!=sha:raise RuntimeError('baseline 완료 기록 변경')
    module.completed(base,'train')
    train,valid,info=module.load_split(original,base)
    if digest(root/'fixed_valid_ids.csv')!=frozen['valid_ids_sha256']:raise RuntimeError('검증 ID 파일 변경')
    if pd.read_csv(root/'fixed_valid_ids.csv',dtype=str).id.tolist()!=valid.id.tolist():raise RuntimeError('검증 순서 변경')
    for name,stat in frozen['model_file_stats'].items():
        f=Path(original['model_dir'])/name
        if [f.stat().st_size,f.stat().st_mtime_ns]!=stat:raise RuntimeError('모델 파일 변경')
    return module,original,frozen,train,valid


def answer_span(full,empty,eos):
    start=0
    while start<min(len(full),len(empty)) and full[start]==empty[start]:start+=1
    suffix=0
    while suffix<min(len(full)-start,len(empty)-start) and full[-1-suffix]==empty[-1-suffix]:suffix+=1
    end=len(full)-suffix
    if start>=end:raise RuntimeError('정답 token span을 찾지 못했습니다.')
    if end>=len(full) or full[end]!=eos:
        raise RuntimeError('정답 바로 뒤 종료 토큰이 예상과 다릅니다. 템플릿 검토 필요; 자동 우회 없음.')
    return start,end


def install_answer_collator(module):
    import copy
    original=module.DataCollator
    class AnswerOnlyCollator(original):
        def __call__(self,batch):
            enc=super().__call__(batch)
            if not self.train:return enc
            if len(batch)!=1:raise ValueError('이 실험은 baseline과 동일하게 batch_size=1')
            empty=[]
            for sample in batch:
                msgs=copy.deepcopy(sample['messages'])
                if msgs[-1]['role']!='assistant':raise RuntimeError('assistant 정답 없음')
                gold=msgs[-1]['content'][0]['text']
                msgs[-1]['content']=[{'type':'text','text':''}]
                empty.append({'messages':msgs,'image':sample['image']})
            empty_enc=super().__call__(empty)
            full=enc['input_ids'][0].tolist();blank=empty_enc['input_ids'][0].tolist()
            start,end=answer_span(full,blank,self.processor.tokenizer.eos_token_id)
            decoded=self.processor.tokenizer.decode(full[start:end],skip_special_tokens=False).strip()
            if decoded!=gold:raise RuntimeError(f'정답 토큰 검증 실패: {decoded!r} != {gold!r}')
            enc['labels'].fill_(-100)
            enc['labels'][0,start:end+1]=enc['input_ids'][0,start:end+1]
            return enc
    module.DataCollator=AnswerOnlyCollator


def audit_targets(module,adapter,train,out):
    import pandas as pd
    rows=[]
    for row in train.head(3).to_dict('records'):
        enc=adapter.encode(row,training=True)
        ids=enc['input_ids'][0].tolist();labels=enc['labels'][0].tolist()
        for i,t in enumerate(ids):
            rows.append({'id':row['id'],'position':i,'token_id':t,
                         'token':adapter.tokenizer.convert_ids_to_tokens(t),'supervised':labels[i]!=-100})
        del enc
    pd.DataFrame(rows).to_csv(out/'supervised_tokens.csv',index=False,encoding='utf-8-sig')


def train_condition(cfg,size):
    import torch
    root=Path(cfg['output_dir']);module,original,frozen,train,valid=validate_inputs(cfg)
    status=root/f'train_{size}_status.json'
    if status.exists():
        old=read(status)
        if old['state']=='completed':
            for name,h in old.get('artifacts',{}).items():
                if digest(Path(old['directory'])/name)!=h:raise RuntimeError('학습 결과 파일 변경')
            print('완료 학습 재사용:',size,flush=True);return
        if old['state']=='blocked_oom' and not cfg['retry_oom']:return
    out=root/f'train_{size}_{time.strftime("%Y%m%d_%H%M%S")}_{uuid.uuid4().hex[:8]}';out.mkdir()
    settings=dict(original);settings['pixel_budget']=640**2;settings['run_dir']=str(out)
    settings['image_policy']='train pixel budget 640 squared'
    settings['loss_mode']=size;settings['loss']='answer_and_eos_only' if size=='answer_only' else original['loss']
    write(out/'training_config.json',settings);write(status,{'state':'running','directory':str(out)})
    try:
        module.gpu_environment(settings,out)
        if size=='answer_only':install_answer_collator(module)
        adapter=module.ModelAdapter(settings,out)
        adapter.add_lora()  # Fresh base and fresh LoRA; no preceding resolution adapter loaded.
        audit_targets(module,adapter,train,out)
        training=module.train_one_epoch(adapter,train,settings,out)
        checkpoint=out/'adapter_epoch1'
        adapter.model.save_pretrained(checkpoint);adapter.processor.save_pretrained(checkpoint)
        # Use fixed inference budget for both sides of serialization check.
        configure_pixels(adapter,cfg['inference_resolution']**2)
        probe=valid.head(min(5,len(valid)))
        before,_=module.evaluate(adapter,probe,'reload_before',out)
        adapter.reload_adapter(checkpoint)
        after,_=module.evaluate(adapter,probe,'reload_after',out)
        keys=['id','answer','raw_output']
        identical=all(all(x[k]==y[k] for k in keys) for x,y in zip(before,after)) and len(before)==len(after)
        write(out/'reload_check.json',{'n':len(probe),'identical_outputs':identical})
        if not identical:raise RuntimeError('저장/재로드 예측 불일치. 결과 검토 필요')
        artifacts={str(p.relative_to(out)):digest(p) for p in out.rglob('*') if p.is_file()}
        write(status,{'state':'completed','directory':str(out),'checkpoint':str(checkpoint),
                      'training':training,'artifacts':artifacts,'reused_baseline':False})
    except torch.cuda.OutOfMemoryError as exc:
        write(status,{'state':'blocked_oom','directory':str(out),'error':str(exc)})
    except BaseException as exc:
        write(status,{'state':'failed_or_interrupted','directory':str(out),'error':str(exc)});raise


def run_resolution(cfg,size):
    import torch
    from peft import PeftModel
    root=Path(cfg['output_dir']);module,original,frozen,train,valid=validate_inputs(cfg)
    old=verified_status(root,size)
    if old['state']=='completed' or (old['state']=='blocked_oom' and not cfg['retry_oom']):
        summarize(cfg);return
    trained=read(root/f'train_{size}_status.json')
    status=root/f'res_{size}_status.json'
    if trained['state']!='completed':
        write(status,{'state':trained['state'],'error':trained.get('error','training incomplete')});summarize(cfg);return
    for name,h in trained.get('artifacts',{}).items():
        if digest(Path(trained['directory'])/name)!=h:raise RuntimeError('학습 artifact 변경')
    out=root/f'eval_train{size}_{time.strftime("%Y%m%d_%H%M%S")}_{uuid.uuid4().hex[:8]}';out.mkdir()
    settings=dict(original);settings['pixel_budget']=cfg['inference_resolution']**2;settings['run_dir']=str(out)
    write(out/'inference_config.json',settings);write(status,{'state':'running','directory':str(out)})
    try:
        module.gpu_environment(settings,out);adapter=module.ModelAdapter(settings,out)
        adapter.model=PeftModel.from_pretrained(adapter.model,trained['checkpoint'],local_files_only=True,is_trainable=False)
        probe=valid.iloc[0].to_dict();probe.pop('answer',None)
        adapter.generate(adapter.encode(probe,training=False));torch.cuda.synchronize()
        _,metrics=module.evaluate(adapter,valid,'valid',out)
        metrics.update({'train_resolution':640,'loss_mode':size,'inference_resolution':cfg['inference_resolution'],
            'checkpoint':trained['checkpoint'],'training_seconds':trained['training']['seconds'],
            'optimizer_updates':trained['training']['updates'],
            'training_peak_allocated_gib':trained['training']['peak_allocated_gib'],
            'reused_baseline':trained['reused_baseline'],'warmup_samples':1})
        write(out/'valid_metrics.json',metrics)
        artifacts={str(p.relative_to(out)):digest(p) for p in out.rglob('*') if p.is_file()}
        write(status,{'state':'completed','directory':str(out),'artifacts':artifacts})
    except torch.cuda.OutOfMemoryError as exc:
        write(status,{'state':'blocked_oom','directory':str(out),'error':str(exc)})
    except BaseException as exc:
        write(status,{'state':'failed_or_interrupted','directory':str(out),'error':str(exc)});raise
    finally:
        with open(root/'CHANGELOG.md','a',encoding='utf-8') as f:f.write(f"\n- train {size}, inference {cfg['inference_resolution']}: {read(status)['state']}\n")
    summarize(cfg)


if __name__=='__main__':
    config=read(sys.argv[1]);stage=sys.argv[2]
    if stage=='prepare':prepare(config)
    elif stage=='summary':summarize(config)
    else:
        is_train=stage.startswith('train_')
        size=stage.removeprefix('train_')
        if size not in config['resolutions']:raise ValueError('설정에 없는 해상도')
        if is_train:train_condition(config,size)
        else:run_resolution(config,size)

'''

In [3]:
def file_hash(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda:f.read(4*1024*1024),b""): h.update(b)
    return h.hexdigest()
base_config=json.loads((BASELINE_RUN_DIR/"run_config.json").read_text(encoding="utf-8"))
if base_config.get("model_id")!="Qwen/Qwen3.5-9B":raise RuntimeError("Qwen3.5-9B baseline을 지정하세요.")
worker_bytes=(BASELINE_RUN_DIR/"task006_worker.py").read_bytes()
worker_raw_hash=hashlib.sha256(worker_bytes).hexdigest()
worker_lf_hash=hashlib.sha256(worker_bytes.replace(bytes([13,10]),bytes([10]))).hexdigest()
if base_config["worker_sha256"] not in (worker_raw_hash,worker_lf_hash):
    raise RuntimeError(f"baseline 코드 내용이 다릅니다. expected={base_config['worker_sha256']}, raw={worker_raw_hash}, LF={worker_lf_hash}. 기존 파일/설정을 수정하지 말고 실행 폴더를 확인하세요.")
if worker_raw_hash!=base_config["worker_sha256"]:
    print("Windows CRLF 줄바꿈 차이만 확인됨. LF 정규화 해시 검증 통과.")
# Detect package changes instead of silently comparing different environments.
freeze=subprocess.check_output([str(ENV_PYTHON),"-m","pip","freeze"],text=True,encoding="utf-8")
old=(BASELINE_RUN_DIR/"requirements.lock.txt").read_text(encoding="utf-8")
if sorted(freeze.splitlines())!=sorted(old.splitlines()):
    raise RuntimeError("baseline 실행 후 패키지 구성이 바뀌었습니다. baseline 전용 환경을 복원한 뒤 실행하세요.")
CFG={"baseline_dir":str(BASELINE_RUN_DIR),"resolutions":RESOLUTIONS,"inference_resolution":INFERENCE_RESOLUTION,"session_tag":SESSION_TAG,
     "baseline_worker_sha256":base_config["worker_sha256"],
     "baseline_config_sha256":file_hash(BASELINE_RUN_DIR/"run_config.json"),
     "train_record_sha256":file_hash(BASELINE_RUN_DIR/"train_complete.json"),
     "audit_record_sha256":file_hash(BASELINE_RUN_DIR/"audit_complete.json"),
     "worker_sha256":hashlib.sha256(RESOLUTION_WORKER.encode()).hexdigest(),
     "environment_sha256":hashlib.sha256(freeze.encode()).hexdigest(),"code_version":"loss-640-672-v1"}
fingerprint=hashlib.sha256(json.dumps(CFG,sort_keys=True).encode()).hexdigest()[:16]
OUTPUT_DIR=PROJECT_DIR/"output/TASK-006-loss"/("LOSS-"+fingerprint)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
CFG.update(output_dir=str(OUTPUT_DIR),retry_oom=RETRY_OOM)
WORKER=OUTPUT_DIR/"resolution_worker.py";CONFIG=OUTPUT_DIR/"resolution_config.json"
compile(RESOLUTION_WORKER,str(WORKER),"exec")
WORKER.write_bytes(RESOLUTION_WORKER.encode("utf-8"))
CONFIG.write_text(json.dumps(CFG,ensure_ascii=False,indent=2),encoding="utf-8")
(OUTPUT_DIR/"requirements.lock.txt").write_text(freeze,encoding="utf-8")

def show_table():
    import csv
    from IPython.display import display,Markdown,FileLink
    p=OUTPUT_DIR/"loss_comparison.csv"
    if not p.exists(): return
    with open(p,encoding="utf-8-sig",newline="") as f: rows=list(csv.DictReader(f))
    columns=[("loss_mode","loss 방식"),("train_resolution","학습 해상도"),("inference_resolution","추론 해상도"),("training_seconds","학습 초"),("status","상태"),("accuracy_pct","Accuracy(%)"),("correct_n","정답 수"),
             ("n","검증 수"),("delta_pp","전체 입력 대비 %p"),("gain","개선"),("loss","악화"),
             ("parse_failure_pct","파싱 실패(%)"),("seconds_per_sample","초/문항"),("peak_allocated_gib","최대 VRAM(GiB)")]
    def fmt(value):
        if value in (None,""): return "—"
        try: return f"{float(value):.3f}" if any(c in str(value) for c in '.eE') else str(value)
        except ValueError: return str(value).replace('|','/')
    text='| '+' | '.join(b for a,b in columns)+' |\n| '+' | '.join('---' for _ in columns)+' |\n'
    for row in rows: text+='| '+' | '.join(fmt(row.get(a,'')) for a,b in columns)+' |\n'
    display(Markdown(text));print("결과 파일:",p);display(FileLink(str(p)))

def run_stage(stage):
    # Same resolution project lock prevents simultaneous notebook runs.
    lock=PROJECT_DIR/"output/TASK-006-loss/gpu_experiment.lock"
    if (BASELINE_RUN_DIR/"running.lock").exists():raise RuntimeError("baseline 작업이 실행 중입니다. 종료 후 실행하세요.")
    try: fd=os.open(lock,os.O_CREAT|os.O_EXCL|os.O_WRONLY)
    except FileExistsError:raise RuntimeError(f"다른 해상도 실험이 실행 중이거나 잠금이 남아 있습니다: {lock}. 실행 프로세스가 없을 때만 잠금을 삭제하세요.")
    process=None
    try:
        with os.fdopen(fd,"w") as f:f.write(str(os.getpid()))
        env=os.environ.copy();env.update(PYTHONIOENCODING="utf-8",PYTHONUNBUFFERED="1")
        with open(OUTPUT_DIR/(str(stage)+".log"),"a",encoding="utf-8") as log:
            process=subprocess.Popen([str(ENV_PYTHON),"-u",str(WORKER),str(CONFIG),str(stage)],
                stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding="utf-8",errors="replace",env=env)
            for line in process.stdout:print(line,end="");log.write(line);log.flush()
            if process.wait()!=0:raise RuntimeError(f"단계 {stage} 실패. {OUTPUT_DIR / (str(stage)+'.log')} 확인")
    except BaseException:
        if process is not None and process.poll() is None:
            process.terminate()
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.kill();process.wait()
        if str(stage) in RESOLUTIONS or str(stage).startswith("train_"):
            status=OUTPUT_DIR/((str(stage) if str(stage).startswith("train_") else "res_"+str(stage))+"_status.json")
            if status.exists():
                rec=json.loads(status.read_text(encoding="utf-8"))
                if rec["state"]=="running":
                    rec["state"]="interrupted";status.write_text(json.dumps(rec,ensure_ascii=False,indent=2),encoding="utf-8")
        raise
    finally:lock.unlink(missing_ok=True)
    show_table()
print("새 실험 결과 폴더:",OUTPUT_DIR)


Windows CRLF 줄바꿈 차이만 확인됨. LF 정규화 해시 검증 통과.
새 실험 결과 폴더: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-loss\LOSS-51ca09225d304a9a


## 3. 기준 입력 확인
checkpoint·코드·패키지·모델 파일·분할·이미지 해시를 확인합니다. 모델 파일 전체를 확인하므로 이 단계는 시간이 걸릴 수 있습니다. 최종 검증은 평가하지 않습니다.

In [4]:
run_stage("prepare")

준비 완료. 저장 LoRA: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006\TASK006-8f2c71505c11284c\train_20260922_115654_f6f6586f\adapter_epoch1 검증: 500
  loss_mode  status  train_resolution  inference_resolution error
  full_text not_run               640                   672      
answer_only not_run               640                   672      


| loss 방식 | 학습 해상도 | 추론 해상도 | 학습 초 | 상태 | Accuracy(%) | 정답 수 | 검증 수 | 전체 입력 대비 %p | 개선 | 악화 | 파싱 실패(%) | 초/문항 | 최대 VRAM(GiB) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| full_text | 640 | 672 | — | not_run | — | — | — | — | — | — | — | — | — |
| answer_only | 640 | 672 | — | not_run | — | — | — | — | — | — | — | — | — |


결과 파일: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-loss\LOSS-51ca09225d304a9a\loss_comparison.csv


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-loss\LOSS-51ca09225d304a9a\loss_comparison.csv

## 4. 조건 1/2 — 전체 입력 loss, 새 기준 학습
설정 셀 `RESOLUTIONS[0]`를 사용합니다. 조건이 완료되면 아래에 누적 비교표가 표시됩니다.

In [5]:
run_stage("train_" + str(RESOLUTIONS[0]))
run_stage(RESOLUTIONS[0])

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'os': 'Windows-10-10.0.26200-SP0', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked; local model files only'}
W0922 14:38:31.221000 7984 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 2/760 [00:01<08:50,  1.43it/s]C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\envs\TASK006_baseline_qwen35\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: Fu

| loss 방식 | 학습 해상도 | 추론 해상도 | 학습 초 | 상태 | Accuracy(%) | 정답 수 | 검증 수 | 전체 입력 대비 %p | 개선 | 악화 | 파싱 실패(%) | 초/문항 | 최대 VRAM(GiB) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| full_text | 640 | 672 | — | not_run | — | — | — | — | — | — | — | — | — |
| answer_only | 640 | 672 | — | not_run | — | — | — | — | — | — | — | — | — |


결과 파일: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-loss\LOSS-51ca09225d304a9a\loss_comparison.csv


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-loss\LOSS-51ca09225d304a9a\loss_comparison.csv

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'os': 'Windows-10-10.0.26200-SP0', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked; local model files only'}
W0922 15:21:18.217000 24640 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 2/760 [00:01<08:42,  1.45it/s]C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\envs\TASK006_baseline_qwen35\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: F

| loss 방식 | 학습 해상도 | 추론 해상도 | 학습 초 | 상태 | Accuracy(%) | 정답 수 | 검증 수 | 전체 입력 대비 %p | 개선 | 악화 | 파싱 실패(%) | 초/문항 | 최대 VRAM(GiB) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| full_text | 640 | 672 | 2532.542 | completed | 91.800 | 459.000 | 500.000 | — | — | — | — | 0.784 | 7.652 |
| answer_only | 640 | 672 | — | not_run | — | — | — | — | — | — | — | — | — |


결과 파일: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-loss\LOSS-51ca09225d304a9a\loss_comparison.csv


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-loss\LOSS-51ca09225d304a9a\loss_comparison.csv

## 5. 조건 2/2 — 정답 부분 loss
설정 셀 `RESOLUTIONS[1]`를 사용합니다. 조건이 완료되면 아래에 누적 비교표가 표시됩니다.

In [6]:
run_stage("train_" + str(RESOLUTIONS[1]))
run_stage(RESOLUTIONS[1])

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'os': 'Windows-10-10.0.26200-SP0', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked; local model files only'}
W0922 15:28:17.828000 11952 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 2/760 [00:01<09:00,  1.40it/s]C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\envs\TASK006_baseline_qwen35\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: F

| loss 방식 | 학습 해상도 | 추론 해상도 | 학습 초 | 상태 | Accuracy(%) | 정답 수 | 검증 수 | 전체 입력 대비 %p | 개선 | 악화 | 파싱 실패(%) | 초/문항 | 최대 VRAM(GiB) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| full_text | 640 | 672 | 2532.542 | completed | 91.800 | 459.000 | 500.000 | — | — | — | — | 0.784 | 7.652 |
| answer_only | 640 | 672 | — | not_run | — | — | — | — | — | — | — | — | — |


결과 파일: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-loss\LOSS-51ca09225d304a9a\loss_comparison.csv


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-loss\LOSS-51ca09225d304a9a\loss_comparison.csv

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'os': 'Windows-10-10.0.26200-SP0', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked; local model files only'}
W0922 16:13:23.287000 20096 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 2/760 [00:01<08:45,  1.44it/s]C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\envs\TASK006_baseline_qwen35\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: F

| loss 방식 | 학습 해상도 | 추론 해상도 | 학습 초 | 상태 | Accuracy(%) | 정답 수 | 검증 수 | 전체 입력 대비 %p | 개선 | 악화 | 파싱 실패(%) | 초/문항 | 최대 VRAM(GiB) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| full_text | 640 | 672 | 2532.542 | completed | 91.800 | 459 | 500 | — | — | — | — | 0.784 | 7.652 |
| answer_only | 640 | 672 | 2658.919 | completed | 94.000 | 470 | 500 | 2.200 | 16.000 | 5.000 | — | 0.883 | 7.652 |


결과 파일: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-loss\LOSS-51ca09225d304a9a\loss_comparison.csv


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-loss\LOSS-51ca09225d304a9a\loss_comparison.csv

## 6. 최종 비교표
OOM은 blocked_oom으로 표시하며 부분 예측을 정확도로 집계하지 않습니다. 다른 오류는 잘못된 조건 비교를 피하기 위해 중단합니다.

In [7]:
run_stage("summary")

  loss_mode    status  train_resolution  inference_resolution   n  correct_n  accuracy  parse_failure_rate  fallback_usage_rate    seconds  seconds_per_sample  peak_allocated_gib  peak_reserved_gib                                                                                                                                checkpoint  training_seconds  optimizer_updates  training_peak_allocated_gib  reused_baseline  warmup_samples  accuracy_pct  gain  loss  delta_pp
  full_text completed               640                   672 500        459     0.918                 0.0                  0.0 391.805877            0.783612            7.652091          11.333984   C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-loss\LOSS-51ca09225d304a9a\train_full_text_20260922_143828_bff2578f\adapter_epoch1       2532.541633                250                     9.904253            False               1          91.8   NaN   NaN       NaN
answer_only completed               640                   

| loss 방식 | 학습 해상도 | 추론 해상도 | 학습 초 | 상태 | Accuracy(%) | 정답 수 | 검증 수 | 전체 입력 대비 %p | 개선 | 악화 | 파싱 실패(%) | 초/문항 | 최대 VRAM(GiB) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| full_text | 640 | 672 | 2532.542 | completed | 91.800 | 459 | 500 | — | — | — | — | 0.784 | 7.652 |
| answer_only | 640 | 672 | 2658.919 | completed | 94.000 | 470 | 500 | 2.200 | 16.000 | 5.000 | — | 0.883 | 7.652 |


결과 파일: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-loss\LOSS-51ca09225d304a9a\loss_comparison.csv


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-loss\LOSS-51ca09225d304a9a\loss_comparison.csv

## 결과 확인

**가장 먼저 `loss_comparison.csv`를 확인하세요.** 같은 표가 각 실행 셀 아래에도 표시됩니다.

| 파일 | 내용 |
|---|---|
| loss_comparison.csv | 2개 loss 조건 상태, Accuracy·정답 수·전체 입력 대비 변화·시간·VRAM |
| eval_train조건_실행시각/valid_predictions.csv | 원출력·예측·정답·파싱 실패·그룹·실제 이미지 크기/시각 토큰 |
| eval_train조건_실행시각/valid_metrics.json | 조건별 지표 원본 |
| paired_predictions.csv | 동일 문항별 정오 비교 |
| changed_answers.csv | 전체 입력 대비 개선·악화 문항만 |
| type_comparison.csv | 기존 임시 문항 유형별 비교 |
| train_조건_실행시각/reload_check.json | 새 학습 모델의 저장/재로드 예측 비교 |
| summary.json | 완료 조건 및 상태 요약 |
| PROJECT_STATUS_update.md / CHANGELOG.md | 반영용 기록. 공식 PROJECT_STATUS는 수정하지 않음 |

- 실제 검증 개수는 기존 분할을 그대로 따릅니다. 새 500개를 추출하지 않습니다.
- 정답 수가 늘어도 작은 차이만으로 채택하지 않습니다. 개선/악화 문항과 비용을 함께 확인하세요.
- 저장/재로드 검사가 실패하면 해당 조건을 완료로 기록하지 않습니다.
- loss 변경 실험입니다. 조건는 학습 640²/추론 672²로 고정됩니다.
- 작성 시 문법/CPU 로직을 검사했습니다. 작성 환경에서 실제 GPU 실행은 하지 않았습니다.
